# Experiment: Multistep Inverse Dynamics + Oracle-Subgoal Goal Reaching

Five environments × maximum horizon `k_max ∈ {1, 2, 4, 8}`. Following Lamb et
al. (arXiv 2207.08229), the inverse head predicts the **first action** `a_t`
from the endpoint pair `(z_t, z_{t+k}, k)`, with `k ~ U{1..k_max}` per training
example. `k_max = 1` is the single-step baseline (bitwise `train.py`).

Caveat under test (flagged, not fixed): the identifiability result assumes
finite actions and deterministic endogenous dynamics; here actions are
continuous, so the endpoints do not determine `a_t` and the MSE head regresses
the conditional mean `E[a_t | z_t, z_{t+k}, k]`. The per-horizon inverse-loss
panel makes any degradation with `k` visible rather than hidden in aggregates.

The notebook answers:

1. **Does the first-action multistep objective still recover the controllable
   DOFs?** — latent effective rank / PCA knee vs. the true controllable dim,
   and a held-out ridge probe `z_t → state` on an independent uniform sample.
2. **Does oracle decoding still reach goals as subgoal spacing grows?** —
   success vs. spacing per `k_max`, under **replanning** (re-encode + decode
   after every step, countdown horizon) and **open-loop** (decode each
   consecutive oracle pair once, execute blind — one first-action per pair,
   the diagnostic lower bound).
3. **Does the distractor config confirm planning ignores the uncontrollable
   dot?** — `1_dot_1_random` vs `single_dot`.

Run `./run.sh` then `./run_eval.sh` first (or the .sub files on the cluster);
the notebook loads whatever runs are present and skips the rest.

In [ ]:
import sys
from pathlib import Path

CWD = Path(".").resolve()
EXPERIMENT_DIR = CWD if (CWD / "config_single_dot_kmax1.yaml").exists() \
                 else CWD / "experiments" / "multistep_inverse"
REPO_ROOT = EXPERIMENT_DIR.parent.parent
for p in (str(REPO_ROOT), str(EXPERIMENT_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

from misc.plot_style import FULL_WIDTH, HALF_WIDTH, palette, apply_matplotlib_style, figure_size
apply_matplotlib_style()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

from train import build_world
import aggregate_results as agg     # reuse effective_rank / ridge_r2 / probe_r2 / collect_run

ENVS = ["single_dot", "2_independent", "1_coupled_pair", "1_dot_1_random", "sprite_xyt"]
KMAXES = [1, 2, 4, 8]
STRUCTURED_ENVS = [e for e in ENVS if e != "sprite_xyt"]

TITLES = {
    "single_dot":     "Single",
    "2_independent":  "Independent",
    "1_coupled_pair": "Coupled",
    "1_dot_1_random": "Distractor",
    "sprite_xyt":     "Sprite (x,y,θ)",
}
TRUE_DIM = {"single_dot": 2, "2_independent": 4, "1_coupled_pair": 2,
            "1_dot_1_random": 2, "sprite_xyt": 3}

# k_max = 1 (single-step baseline) → blue; longer horizons → red.
K_COLOR = {1: palette["Dark Blue"], 2: palette["Med Blue"],
           4: palette["Med Red"], 8: palette["Dark Red"]}
ENV_COLOR = {"single_dot": palette["Dark Grey"], "2_independent": palette["Dark Blue"],
             "1_coupled_pair": palette["Dark Red"], "1_dot_1_random": palette["Med Purple"],
             "sprite_xyt": palette["Med Red"]}
MODE_STYLE = {"replanning": "-", "open_loop": "--"}

def run_name(env, kmax):
    return f"{env}_kmax{kmax}"

In [ ]:
RESULTS_DIR = EXPERIMENT_DIR / "results"

runs = {}     # (env, k_max) -> {cfg, history, embeddings, goal}
rows = []     # per-run summary via aggregate_results.collect_run (== summary.csv row)
for env in ENVS:
    for kmax in KMAXES:
        name = run_name(env, kmax)
        run_dir = RESULTS_DIR / name
        if not (run_dir / "config.yaml").exists():
            continue
        entry = {"cfg": yaml.safe_load((run_dir / "config.yaml").read_text())}
        for key, fname in [("history", "train_history.pt"),
                           ("embeddings", "embeddings.pt"),
                           ("goal", "goal_reaching.pt")]:
            if (run_dir / fname).exists():
                entry[key] = torch.load(run_dir / fname, map_location="cpu", weights_only=False)
        runs[(env, kmax)] = entry
        row = agg.collect_run(run_dir)
        if row is not None:
            rows.append(row)
        print(f"  loaded {name}" + ("" if "goal" in entry else "   (no goal_reaching.pt)"))

df = pd.DataFrame(rows).set_index("run") if rows else pd.DataFrame()
HAS_RESULTS = len(runs) > 0
HAS_GOAL = any("goal" in e for e in runs.values())
present_envs = [e for e in ENVS if any((e, k) in runs for k in KMAXES)]
if not HAS_RESULTS:
    print("No results under", RESULTS_DIR, "— run ./run.sh and ./run_eval.sh first.")

In [ ]:
def pca_expl_var(Z, eps=1e-12):
    Zc = Z - Z.mean(axis=0)
    _, S, _ = np.linalg.svd(Zc, full_matrices=False)
    ev = np.maximum((S ** 2) / max(1, Z.shape[0] - 1), eps)
    return ev / ev.sum()

def final_latents(entry):
    # Concatenated (z_t, z_{t+1}) from the last eval snapshot (encoder geometry).
    snaps = entry["embeddings"]["snapshots"]
    snap = snaps[max(snaps)]
    return torch.cat([snap["z_t"], snap["z_tp1"]]).numpy()

def series(env, col):
    # (k_maxes, values) for a df column across the present runs of one env.
    ks, vals = [], []
    for kmax in KMAXES:
        name = run_name(env, kmax)
        if name in df.index and pd.notna(df.loc[name, col]):
            ks.append(kmax); vals.append(float(df.loc[name, col]))
    return ks, vals

def goal_series(entry, mode, key):
    g = entry["goal"]
    sp = g["spacings"]
    return sp, [g["summary"][mode][m][key] for m in sp]

## Training converged for every k_max
Inverse loss is the anti-collapse signal; forward loss tracks latent prediction.

In [ ]:
if HAS_RESULTS:
    ncol = len(present_envs)
    fig, axes = plt.subplots(2, ncol, figsize=figure_size(FULL_WIDTH, height=2.6),
                             squeeze=False, sharex=True)
    for j, env in enumerate(present_envs):
        for i, key in enumerate(["fwd", "inv"]):
            ax = axes[i][j]
            for kmax in KMAXES:
                e = runs.get((env, kmax))
                if not e or "history" not in e:
                    continue
                tr, ev = e["history"]["train"], e["history"]["eval"]
                ax.semilogy(tr["epoch"], tr[key], color=K_COLOR[kmax], alpha=0.25, lw=0.8)
                ax.semilogy(ev["epoch"], ev[key], color=K_COLOR[kmax], lw=1.1, label=f"k_max={kmax}")
            ax.grid(True, which="both", alpha=0.3)
            if i == 0:
                ax.set_title(TITLES[env], pad=3)
            if j == 0:
                ax.set_ylabel(("Forward" if key == "fwd" else "Inverse") + " loss")
    for ax in axes[1]:
        ax.set_xlabel("Epoch")
    axes[0][-1].legend(fontsize=6, loc="upper right")
    fig.suptitle("Training losses — train (faint) vs eval (solid), colored by k_max")
    fig.tight_layout()
    plt.show()

## Inverse loss per horizon k — the multimodality instrumentation

With continuous actions the endpoints do not determine `a_t`; the regression
optimum is the conditional mean, whose residual grows with `k` (more action
sequences share endpoints). This panel shows the final eval inverse loss at
each `k` inside every run, so degradation at larger horizons is visible
directly rather than hidden in the aggregate.

In [ ]:
if HAS_RESULTS:
    ncol = len(present_envs)
    fig, axes = plt.subplots(1, ncol, figsize=figure_size(FULL_WIDTH, ratio=0.26),
                             squeeze=False, sharey=False)
    for j, env in enumerate(present_envs):
        ax = axes[0][j]
        for kmax in KMAXES:
            e = runs.get((env, kmax))
            if not e or "history" not in e:
                continue
            per_k_hist = e["history"]["eval"].get("inv_per_k")
            if not per_k_hist:
                continue
            per_k = per_k_hist[-1]
            ks = sorted(per_k)
            ax.semilogy(ks, [per_k[k] for k in ks], "-o", color=K_COLOR[kmax],
                        label=f"k_max={kmax}")
        ax.set_title(TITLES[env]); ax.set_xlabel("horizon k")
        ax.set_xticks(KMAXES)
        ax.grid(True, which="both", alpha=0.3)
    axes[0][0].set_ylabel("eval inverse loss at k")
    axes[0][-1].legend(fontsize=6)
    fig.suptitle("Final eval inverse loss per horizon k (first-action target)")
    fig.tight_layout()
    plt.show()

## Q1 — Controllable DOFs are still recovered as k_max grows

If the first-action multistep objective keeps the encoder faithful, the latent
**effective rank** should sit near the true controllable dim across `k_max`,
the probe **R²(controllable)** should stay ≈ 1, and **R²(uncontrollable)**
(distractor only) should stay ≈ 0. The probe is a held-out ridge fit of
`z_t → state` on the eval split — an independent uniform sample of states.

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 3, figsize=figure_size(FULL_WIDTH, ratio=0.32))

    ax = axes[0]                                             # effective rank vs k_max
    for env in present_envs:
        ks, vals = series(env, "eff_rank")
        if ks:
            ax.plot(ks, vals, "-o", color=ENV_COLOR[env], label=TITLES[env])
            ax.axhline(TRUE_DIM[env], color=ENV_COLOR[env], ls=":", lw=0.7, alpha=0.7)
    ax.set_title("Latent effective rank"); ax.set_xlabel("k_max")
    ax.set_xscale("log", base=2); ax.set_xticks(KMAXES); ax.set_xticklabels(KMAXES)
    ax.grid(True, alpha=0.3); ax.legend(fontsize=6)
    ax.text(0.02, 0.02, "dotted = true dim", transform=ax.transAxes, fontsize=6, color="0.4")

    for ax, col, title in [(axes[1], "r2_ctrl", "Probe R²  (controllable)"),
                           (axes[2], "r2_unctrl", "Probe R²  (uncontrollable)")]:
        drew = False
        for env in present_envs:
            ks, vals = series(env, col)
            if ks:
                ax.plot(ks, vals, "-o", color=ENV_COLOR[env], label=TITLES[env]); drew = True
        ax.set_title(title); ax.set_xlabel("k_max")
        ax.set_xscale("log", base=2); ax.set_xticks(KMAXES); ax.set_xticklabels(KMAXES)
        ax.set_ylim(-0.05, 1.05); ax.grid(True, alpha=0.3)
        if not drew:
            ax.text(0.5, 0.5, "n/a", transform=ax.transAxes, ha="center", color="0.6")
    fig.suptitle("z_t → state probe (held-out R²) and latent rank vs k_max")
    fig.tight_layout()
    plt.show()

In [ ]:
# PCA spectra: structured worlds (clean integer dims) × k_max. Knee should stay put.
import matplotlib.gridspec as gridspec
PC_CUTOFF = 16
structured_present = [e for e in STRUCTURED_ENVS if any((e, k) in runs for k in KMAXES)]
if structured_present:
    nrow, ncol = len(structured_present), len(KMAXES)
    fig = plt.figure(figsize=figure_size(FULL_WIDTH, height=0.72 * nrow + 0.5))
    gs = gridspec.GridSpec(nrow, ncol, figure=fig, hspace=0.25, wspace=0.12,
                           left=0.12, right=0.99, bottom=0.12, top=0.9)
    for r, env in enumerate(structured_present):
        for c, kmax in enumerate(KMAXES):
            ax = fig.add_subplot(gs[r, c])
            e = runs.get((env, kmax))
            if not e or "embeddings" not in e:
                ax.set_axis_off(); continue
            evr = pca_expl_var(final_latents(e))
            n_show = min(PC_CUTOFF, len(evr))
            ax.bar(np.arange(1, n_show + 1), evr[:n_show], color=K_COLOR[kmax], width=0.7)
            ax.axvline(TRUE_DIM[env] + 0.5, color=palette["Dark Grey"], ls="--", lw=0.8)
            ax.set_xlim(0.5, PC_CUTOFF + 0.5); ax.set_ylim(0, None)
            ax.set_xticks([1, 5, 10, 15]); ax.tick_params(labelsize=6)
            if r == 0:
                ax.set_title(f"k_max = {kmax}", pad=3)
            if c == 0:
                ax.set_ylabel(f"{TITLES[env]}\n(dim {TRUE_DIM[env]})", fontsize=7)
            if r == nrow - 1:
                ax.set_xlabel("PC", fontsize=7)
    fig.suptitle("PCA spectra of learned latents — dashed = true controllable dim")
    plt.show()

## Q2 — Oracle decoding vs subgoal spacing, replanning vs open-loop

Solid = **replanning** (re-encode + decode toward the active subgoal after
every executed step, queried horizon counting down to 1). Dashed =
**open-loop** (one decoded first-action per consecutive oracle pair, executed
blind). Open-loop at spacing m > 1 can only recover each gap's first action,
so it is expected to collapse with spacing; replanning is the practical mode.

In [ ]:
if HAS_GOAL:
    envs_g = [e for e in present_envs if any("goal" in runs.get((e, k), {}) for k in KMAXES)]
    fig, axes = plt.subplots(1, len(envs_g), figsize=figure_size(FULL_WIDTH, ratio=0.26),
                             squeeze=False, sharey=True)
    for j, env in enumerate(envs_g):
        ax = axes[0][j]
        for kmax in KMAXES:
            e = runs.get((env, kmax))
            if not e or "goal" not in e:
                continue
            for mode in e["goal"]["modes"]:
                sp, succ = goal_series(e, mode, "success_rate")
                ax.plot(sp, succ, MODE_STYLE[mode], color=K_COLOR[kmax], marker="o", ms=2.5,
                        label=f"k_max={kmax}" if mode == "replanning" else None)
        ax.set_title(TITLES[env]); ax.set_xlabel("subgoal spacing")
        ax.set_xscale("log", base=2); ax.set_xticks(sp); ax.set_xticklabels(sp)
        ax.set_ylim(-0.03, 1.03); ax.grid(True, alpha=0.3)
    axes[0][0].set_ylabel("success rate")
    axes[0][-1].legend(fontsize=6, loc="lower left",
                       title="solid: replanning\ndashed: open-loop", title_fontsize=6)
    fig.suptitle("Goal reaching: success vs subgoal spacing (higher = better)")
    fig.tight_layout()
    plt.show()
else:
    print("No goal_reaching.pt found — run ./run_eval.sh to populate Q2/Q3.")

In [ ]:
if HAS_GOAL:                                # final controllable position error (px)
    envs_g = [e for e in present_envs if any("goal" in runs.get((e, k), {}) for k in KMAXES)]
    fig, axes = plt.subplots(1, len(envs_g), figsize=figure_size(FULL_WIDTH, ratio=0.26),
                             squeeze=False, sharey=True)
    thr = 3.0
    for j, env in enumerate(envs_g):
        ax = axes[0][j]
        for kmax in KMAXES:
            e = runs.get((env, kmax))
            if not e or "goal" not in e:
                continue
            for mode in e["goal"]["modes"]:
                sp, perr = goal_series(e, mode, "pos_err_mean")
                ax.plot(sp, perr, MODE_STYLE[mode], color=K_COLOR[kmax], marker="o", ms=2.5,
                        label=f"k_max={kmax}" if mode == "replanning" else None)
            thr = e["goal"]["eval"]["success_threshold"]
        ax.axhline(thr, color="0.4", ls=":", lw=0.7)
        ax.set_title(TITLES[env]); ax.set_xlabel("subgoal spacing")
        ax.set_xscale("log", base=2); ax.set_xticks(sp); ax.set_xticklabels(sp)
        ax.set_yscale("log"); ax.grid(True, which="both", alpha=0.3)
    axes[0][0].set_ylabel("final pos. error (px)")
    axes[0][-1].legend(fontsize=6, loc="upper left")
    fig.suptitle("Goal reaching: final controllable error (dotted = success threshold)")
    fig.tight_layout()
    plt.show()

## Q3 — The distractor is ignored

`1_dot_1_random` (Distractor) vs `single_dot` (Single), replanning mode. If the
planner ignores the uncontrollable dot, the controllable success should
**track Single**, the random dot's error should stay at **chance**
(independent of `k_max` / spacing), and its probe **R²(uncontrollable) ≈ 0**.

In [ ]:
have_distr = any(("1_dot_1_random", k) in runs and "goal" in runs[("1_dot_1_random", k)]
                 for k in KMAXES)
if not have_distr:
    print("Distractor goal_reaching.pt not found — run ./run_eval.sh for 1_dot_1_random.")
else:
    fig, axes = plt.subplots(1, 3, figsize=figure_size(FULL_WIDTH, ratio=0.3))

    ax = axes[0]                            # (a) success: Distractor (solid) vs Single (dashed)
    for kmax in KMAXES:
        for env, ls, lab in [("1_dot_1_random", "-", "Distractor"), ("single_dot", "--", "Single")]:
            e = runs.get((env, kmax))
            if not e or "goal" not in e or "replanning" not in e["goal"]["modes"]:
                continue
            sp, succ = goal_series(e, "replanning", "success_rate")
            ax.plot(sp, succ, ls, color=K_COLOR[kmax], marker="o", ms=2.5,
                    label=f"{lab} k_max={kmax}" if kmax in (1, 8) else None)
    ax.set_title("Success — Distractor vs Single"); ax.set_ylim(-0.03, 1.03)
    ax.set_xscale("log", base=2); ax.set_xticks(sp); ax.set_xticklabels(sp)
    ax.set_xlabel("subgoal spacing"); ax.set_ylabel("success rate")
    ax.grid(True, alpha=0.3); ax.legend(fontsize=5.5)

    ax = axes[1]                            # (b) controllable vs uncontrolled error (distractor)
    for kmax in KMAXES:
        e = runs.get(("1_dot_1_random", kmax))
        if not e or "goal" not in e or "replanning" not in e["goal"]["modes"]:
            continue
        sp, ctrl = goal_series(e, "replanning", "pos_err_mean")
        _, unc = goal_series(e, "replanning", "unctrl_err_mean")
        ax.plot(sp, ctrl, "-o", color=K_COLOR[kmax], ms=2.5,
                label=f"ctrl k_max={kmax}" if kmax in (1, 8) else None)
        ax.plot(sp, unc, ":s", color=K_COLOR[kmax], ms=2.5,
                label=f"random k_max={kmax}" if kmax in (1, 8) else None)
    ax.set_title("Distractor error: controlled vs random dot")
    ax.set_xscale("log", base=2); ax.set_xticks(sp); ax.set_xticklabels(sp)
    ax.set_xlabel("subgoal spacing"); ax.set_ylabel("pos. error (px)")
    ax.grid(True, alpha=0.3); ax.legend(fontsize=5.5)

    ax = axes[2]                            # (c) probe R²: controllable vs uncontrollable
    ks_c, r2c = series("1_dot_1_random", "r2_ctrl")
    ks_u, r2u = series("1_dot_1_random", "r2_unctrl")
    if ks_c:
        ax.plot(ks_c, r2c, "-o", color=palette["Dark Blue"], label="controllable")
    if ks_u:
        ax.plot(ks_u, r2u, "-o", color=palette["Med Purple"], label="uncontrollable")
    ax.set_title("Distractor probe R²"); ax.set_ylim(-0.05, 1.05)
    ax.set_xscale("log", base=2); ax.set_xticks(KMAXES); ax.set_xticklabels(KMAXES)
    ax.set_xlabel("k_max"); ax.grid(True, alpha=0.3); ax.legend(fontsize=6)
    fig.suptitle("Distractor: the uncontrollable dot is ignored by decoding")
    fig.tight_layout()
    plt.show()

## Summary table
Same numbers as `results/summary.csv` (from `aggregate_results.py`), recomputed here.

In [ ]:
if not df.empty:
    base = ["world", "k_max", "true_dim", "eval_fwd", "eval_inv",
            "eff_rank", "r2_ctrl", "r2_unctrl"]
    def spacing_cols(prefix):
        cols = [c for c in df.columns if c.startswith(prefix + "@")]
        return sorted(cols, key=lambda c: int(c.split("@")[1]))
    cols = ([c for c in base if c in df.columns] + spacing_cols("inv")
            + spacing_cols("succ") + spacing_cols("ol_succ"))
    display(df[cols].sort_values(["world", "k_max"]).style.format(precision=3, na_rep="—"))